# Stage 5.05 — validate, analyze, and export

Run validation and analysis for both 5A and 5B. When the 5A0 gate is closed, 5A analysis still produces the selected operating point and the final observations without any episode results.

In [1]:
# 5A analysis: produces the selected operating point even with an empty results file.
import os, subprocess, sys
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"
ANALYSIS.mkdir(exist_ok=True)
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.analyze_stage5",
    "--phase", "a",
    "--results", str(OUT / "stage5a_episode_results.csv"),
    "--output-dir", str(ANALYSIS),
], cwd=R, check=True)


PASS: Stage 5A gate closed; selected coverage remains native 8


CompletedProcess(args=['/opt/tljh/user/bin/python3.12', '-m', 'async_vla_benchmark.scripts.analyze_stage5', '--phase', 'a', '--results', '/home/jupyter-smtm-5bba/stage5/stage5a_episode_results.csv', '--output-dir', '/home/jupyter-smtm-5bba/stage5/analysis'], returncode=0)

In [2]:
# Build the conditional Stage 5B manifest from the selected operating point.
import json, subprocess
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
prov = json.loads((OUT / "stage5_provenance.json").read_text())
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.make_stage5b_manifest",
    "--output", str(OUT / "stage5b_manifest.csv"),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
    "--git-sha", prov["git_sha"],
    "--libero-plus-git-sha", prov["libero_plus_git_sha"],
], cwd=R, check=True)


PASS empty Stage 5B manifest: 5A operating point does not warrant a rerun


CompletedProcess(args=['/opt/tljh/user/bin/python3.12', '-m', 'async_vla_benchmark.scripts.make_stage5b_manifest', '--output', '/home/jupyter-smtm-5bba/stage5/stage5b_manifest.csv', '--selected', '/home/jupyter-smtm-5bba/stage5/analysis/stage5a_selected_operating_point.json', '--git-sha', 'd79576f25087e6154f67b81d5a1b710ccb96528c', '--libero-plus-git-sha', '4976dc30028e805ff8094b55501d532c48fec182'], returncode=0)

In [3]:
# 5A validation (now the selected operating point exists).
import os, subprocess, sys
from pathlib import Path
R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.validate_stage5",
    "--phase", "a",
    "--manifest", str(OUT / "stage5a_manifest.csv"),
    "--output-dir", str(OUT),
    "--audit", str(OUT / "stage5_openvla_coverage_capability_audit.json"),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
], cwd=R, check=True)


{
  "phase": "5a",
  "manifest_rows": 0,
  "result_rows": 0,
  "status": "pass",
  "errors": [],
  "manifest_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
}


CompletedProcess(args=['/opt/tljh/user/bin/python3.12', '-m', 'async_vla_benchmark.scripts.validate_stage5', '--phase', 'a', '--manifest', '/home/jupyter-smtm-5bba/stage5/stage5a_manifest.csv', '--output-dir', '/home/jupyter-smtm-5bba/stage5', '--audit', '/home/jupyter-smtm-5bba/stage5/stage5_openvla_coverage_capability_audit.json', '--selected', '/home/jupyter-smtm-5bba/stage5/analysis/stage5a_selected_operating_point.json'], returncode=0)

In [4]:
# 5B validation and analysis (conditional; no-ops when the gate is closed)
import os, subprocess, sys
from pathlib import Path
R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.validate_stage5",
    "--phase", "b",
    "--manifest", str(OUT / "stage5b_manifest.csv"),
    "--output-dir", str(OUT),
    "--selected", str(ANALYSIS / "stage5a_selected_operating_point.json"),
], cwd=R, check=True)

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.analyze_stage5",
    "--phase", "b",
    "--results", str(OUT / "stage5b_episode_results.csv"),
    "--output-dir", str(ANALYSIS),
], cwd=R, check=True)


{
  "phase": "5b",
  "manifest_rows": 0,
  "status": "pass",
  "result_rows": 0,
  "errors": [],
  "manifest_sha256": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"
}


PASS: no Stage 5B results to analyze (conditional on 5A operating point)


CompletedProcess(args=['/opt/tljh/user/bin/python3.12', '-m', 'async_vla_benchmark.scripts.analyze_stage5', '--phase', 'b', '--results', '/home/jupyter-smtm-5bba/stage5/stage5b_episode_results.csv', '--output-dir', '/home/jupyter-smtm-5bba/stage5/analysis'], returncode=0)

In [5]:
import hashlib
from pathlib import Path
OUT = Path.home() / "stage5"
ANALYSIS = OUT / "analysis"

expected = ["stage5a_selected_operating_point.json", "STAGE_5A_OBSERVATIONS.md"]
missing = [x for x in expected if not (ANALYSIS / x).exists()]
if missing:
    raise SystemExit(f"missing analysis artifacts: {missing}")

print("Stage 5 analysis artifacts:")
for p in sorted(ANALYSIS.iterdir()):
    print(" ", p.name)


Stage 5 analysis artifacts:
  STAGE_5A_OBSERVATIONS.md
  stage5a_selected_operating_point.json
